In [ ]:
forget()

load("Library.sage")

R.<a_11,a_22,a_31,a_12> = LaurentPolynomialRing(QQ);
R_poly = R.polynomial_ring();
Q.<t> = PolynomialRing(QQ);

LG = a_11^-1 * (a_11 + ((1 + a_22) * a_31^-1 + a_22 * a_12^-1) * (a_31 + (1 + a_12 * a_22 + a_22) * a_22^-1)^2 )^2;

LG_mutated = (a_11^-4*a_22^-2*a_31^-2*a_12^-1) * (a_11*a_22 + a_11*a_31 + a_31) * \
(a_11*a_22 + a_11*a_31 + a_11 + 1)^2 * (a_11^2*a_22*a_31*a_12 + 1)^2;

LG_monomial_matrix = matrix(ZZ, [[0, 0, 0, 1],
                                 [1, 0, 0, 0],
                                 [0, 1, 0, 0],
                                 [0, 0, 1, 0]]);
LG_mutation_factor = (a_11*a_22 + a_11*a_31 + a_31) * (a_11*a_22 + a_11*a_31 + a_11 + 1)^2;
LG_monomial_change = GL_action(LG, LG_monomial_matrix);
#LG_mutated_alt = laurent_polynomial_mutation(LG_monomial_change, LG_mutation_factor);


## Newton polytope of the Landau--Ginzburg model

polytope = newton_polytope(LG_mutated);
polyhedron = Polyhedron(polytope.vertices(), base_ring=ZZ);

integral_points = polyhedron.integral_points();
interior_points = polytope.interior_points();

edges = polytope.edges();
faces = polytope.faces(2);
facets = polytope.facets();


## Print period sequences

#print("Period sequences:");
#print(period_series_truncated(5, [1,2], 5).coefficients(sparse = false));
#print(period_series_truncated(5, [2,2], 5).coefficients(sparse = false));
#print(period_sequence(LG_mutated, 6));


## Compute the edge polynomials

#print("Edge polynomials:");
#edge_poly_list = [];
#for edge in edges:
#    edge_polyhedron = Polyhedron(edge.vertices(), base_ring=ZZ);
#    edge_polynomial = 0;
#    integral_points_number = len(edge_polyhedron.integral_points());
#    for i in range(integral_points_number):
#        point = edge_polyhedron.integral_points()[i];
#        monomial = R.monomial(*list(point));
#        coeff = LG_mutated.monomial_coefficient(monomial);
#        edge_polynomial += coeff*t^i;
#    edge_poly_list.append(factor(edge_polynomial));
#print(edge_poly_list);


## Try to find a smooth fine regular star triangulation
## by bruteforcing the nearest mutations

#find_smooth_FRST(LG, 1)


## Check that the choosen FRS triangulation
## actually defines a smooth toric variety

#triang_vertices = matrix(ZZ, [[-1,  0,  0,  0,  0,  0,  0,  1,  1],
#                              [0,  -1,  0,  0,  0,  1,  1,  -1,  -1],
#                              [0,  -1,  0,  0,  1,  0,  1,  -1,  0],
#                              [2,  2,  -1,  1,  -1,  -1,  -2,  0,  -1]]);
#triang_simplices = matrix(ZZ, [[0,  0,  0,  0,  1,  1,  1,  1,  1,  1],
#                               [1,  1,  1,  1,  2,  2,  2,  3,  3,  3],
#                               [2,  2,  3,  3,  4,  5,  6,  4,  5,  6],
#                               [4,  5,  4,  5,  6,  6,  7,  6,  6,  7],
#                               [6,  6,  6,  6,  8,  7,  8,  8,  7,  8]]);
#star_origin = 1;

#newton_polar = LatticePolytope(triang_vertices.columns());
#print("Does the convex hull actually coincides with the dual polytope:")
#print(newton_polar.polyhedron().integral_points() == \
#      polytope.polar().polyhedron().integral_points());
#newton_polar_length = len(newton_polar.vertices());
#pointConf = PointConfiguration(newton_polar.vertices(), star = star_origin);
#triangulation = Triangulation(triang_simplices.columns(), pointConf);
#print("Is the associated toric variety smooth:");
#print(ToricVariety(triangulation.fan()).is_smooth());


## Compute facet polynomials

#face_polynomial(LG_mutated, 3)


## Print vertices of 3d Minkowski summands in PolyMake format

#c = -1;
#for [P,Q,S] in face_minkowski_polytopes(LG_mutated, 3, refined = False):
#    if (P.dim() != 3) : continue;
#    c += 1;
#    print("$F_" + str(c) + " = new Matrix(" + \
#          str([list([1] + list(vector(v))) for v in P.vertices()]) + ");\n");


## Check that all irreducible 2d Minkowski summands
## are hollow triangles of lattice width 1

#bad = 0;
#for [P,Q,S] in face_minkowski_polytopes(LG_mutated, 2, refined = False):
#    if (P.dim() != 2) : continue;
#    if (len(P.vertices()) != 3) :
#        bad = 1; break;
#    if (len(P.interior_points()) > 0) :
#        bad = 1; break;
#    # We want to exclude the hollow triangle of lattice width 2
#    if (P.polyhedron().volume() == 2) :
#        if (max(P.facet_constants()) != 4):
#            bad = 1; break;
#if not (bad):
#    print("OK!");
#else:
#    print("FAIL!");
   
    
## Check the smoothness of intersections of
## irreducible components of facet polynomials

#bad = 0;
#for List in face_minkowski_polytopes(LG_mutated, 3, refined = True):
#    L = len(List);
#    par = List[0][-2].parent();
#    T.<X_0, X_1, X_2> = toric_varieties.torus(3);
#
#    for I in Subsets(range(L)):
#        if (len(I) < 1) : continue;
#        gens_list = [];
#        component_list = [];
#
#        for i in list(I): gens_list.append((List[i])[-2]);
#        P = T.subscheme(par.ideal(gens_list).radical());
#           
#        # Check that irreducible components of the intersection are smooth
#        for PP in P.irreducible_components():
#            component_list.append(PP.defining_polynomials());
#            II = par.ideal(PP.defining_polynomials()).radical();
#            jacobian_list = [];
#            for poly in II.gens():
#                gradient = [];
#                for g in par.gens(): gradient.append(poly.derivative(g));
#                jacobian_list.append(gradient);
#            jacobian_matrix = matrix(par, jacobian_list);
#            d = II.dimension();
#            if (d < 1) : continue;
#            c = 3 - d;
#            if (c != len(I)) :
#                print("Warning: non-expected codimension.");
#                print("Codimension: " + str(c));
#                print("Number of generators of the radical: " + str(len(II.gens())));
#                print("Number of intersecting hypersurfaces: " + str(len(I)));
#
#            singular_list = gens_list + jacobian_matrix.minors(c);
#            singular_ideal = par.ideal(singular_list);
#            if (singular_ideal.radical().gens().count(1) == 0):
#                bad = 1;
#
#        # Check that the irreducible component of the intersection do not intersect
#        for J in Subsets(range(len(component_list))):
#            if (len(J) < 2) : continue;
#            print("Warning: we actually have a reducible intersection.");
#            intersection_list = [];
#            for j in list(J): intersection_list += component_list[j];
#            intersection_ideal = par.ideal(intersection_list);
#            if (intersection_ideal.radical().gens().count(1) == 0):
#                bad = 1;
#
#if not (bad):
#    print("OK!");
#else:
#    print("FAIL!");


## For all facet polynomial print generators of ideals
## of intersections of its irreducible components
## while omitting linear generators and cases when
## there is only one non-linear generator
## (i.e., the only cases when the rationality
## over Q of these intersections is not immediate)

#bad = 0;
#for List in face_minkowski_polytopes(LG_mutated, 3, refined = True):
#    L = len(List);
#    par = List[0][-2].parent();
#    par_poly = par.polynomial_ring();
#    frac_field = FractionField(par_poly);
#    T.<X_0, X_1, X_2> = toric_varieties.torus(3);
#
#    for I in Subsets(range(L)):
#        if (len(I) < 2) : continue;
#        gens_list = [];
#        component_list = [];
#
#        for i in list(I): gens_list.append((List[i])[-2]);
#        P = T.subscheme(par.ideal(gens_list).radical());
#            
#        # Check that irreducible components of the intersection are rational over Q
#        for PP in P.irreducible_components():
#            component_list.append(PP.defining_polynomials());
#            II = par.ideal(PP.defining_polynomials()).radical();
#            reduced_list = [];
#            for poly in II.gens():
#                poly_reduced = par_poly(frac_field(poly).numerator());
#                if (poly_reduced.degree() != 1) : reduced_list.append(poly);
#            if (len(reduced_list) > 1): print(reduced_list);
#
#        # Check that the irreducible component of the intersection do not intersect
#        for J in Subsets(range(len(component_list))):
#            if (len(J) < 2) : continue;
#            print("Warning: we actually have a reducible intersection.");
#            intersection_list = [];
#            for j in list(J): intersection_list += component_list[j];
#            intersection_ideal = par.ideal(intersection_list);
#            if (intersection_ideal.radical().gens().count(1) == 0):
#                print("Warning: components of the intersection are not disjoint.");


## Try to naively present facet polynomials
## in the form F(X_0, X_1) * X_2 + G(X_0, X_1) = 0,
## and check that components of
## F(X_0, X_1) = G(X_0, X_1) = 0 are defined over Q.

#for [P,Q,R] in face_minkowski_polytopes(LG_mutated, 3, refined = False):
#    par = Q.parent();
#    par_poly = par.polynomial_ring();
#    frac_field = FractionField(par_poly);
#    poly_reduced = par_poly(frac_field(Q).numerator());
#        
#    if (newton_polytope(Q).dim() != 3) : continue;
#                    
#    is_naive_decomposition_successful = 0;
#
#    for g in range(par_poly.ngens()):
#        gen_list = list(par_poly.gens());
#        gen_list[g] = 0;
#        max_monomial = gcd(poly_reduced.monomials())
#        poly_reduced_min = par_poly(poly_reduced / max_monomial);
#                        
#        G = poly_reduced_min(gen_list);
#        F_pre = poly_reduced_min - G;
#        F_deg = F_pre.degree(par_poly.gen(g));
#        F = F_pre / (par_poly.gen(g)^F_deg);
#        if not (F in par_poly) : continue;
#                        
#        if ((F == 0) or (G == 0)): print("Warning! F = 0 or G = 0."); continue;
#
#        is_naive_decomposition_successful = 1;
#                        
#        are_all_generators_linear = 1;
#                        
#        for J in par_poly.ideal([F,G]).radical().primary_decomposition():
#            are_generators_linear = 1;
#            for gen in J.gens():
#                if (gen.degree() != 1) :
#                    are_generators_linear = 0;
#                    break;
#            if not (are_generators_linear):
#                print("var: " + str(par_poly.gen(g)))
#                print(J.gens());
#                are_all_generators_linear = 0;
#
#        if not (are_all_generators_linear) :
#            print("Warning! Not all generators are linear.");
#        else:
#            print("OK!");
#        break;
#                              
#    if not (is_naive_decomposition_successful) :
#        print("No naive decomposition: " + str([Q,R]));
#        print("Facet normals: ");
#        print(list(newton_polytope(Q).facet_normals()));
#        print("\n")

## If there is no naive decomposition,
## this could be achieved by applying some
## unimodular change of coordinates placing
## the corresponding facet hyperplane
## at z = 0 (see PolyMake computations)

## The similar computation was done
## in QuiverFlagZeroLoci.ipynb